# ActSSM (ActMamba V3): Unified State-Space Architecture
**Task**: Temporal Action Segmentation (TAS) + Long-Term Anticipation (LTA)  
**Supported Datasets**: GTEA, 50Salads, Breakfast

This notebook is pre-configured to run training and evaluation for the **ActSSM** architecture using separate mask tokens for TAS and LTA without positional encoding.

### Instructions:
1. **Upload Code**: Upload the `ActSSM-DualToken` folder as a Kaggle Dataset.
2. **Attach Datasets**: Attach your chosen dataset (GTEA, 50Salads, or Breakfast) to this notebook.
3. **Configure Variables**: Set the `DATASET`, `SPLIT`, `PROJECT_SRC`, and `DATASET_SRC` variables in **Step 2** to match your Kaggle setup.

## Step 1: Install Dependencies
This cell uninstalls conflicting versions, cleans PyPI caches, installs build dependencies, and compiles `causal-conv1d` and `mamba-ssm` from source.
*(Note: Compiling `mamba-ssm` takes about 5 to 10 minutes. This is normal.)*

In [ ]:
# 1a. Clean slate
!pip uninstall -y mamba-ssm causal-conv1d
!pip cache purge

# 1b. Install build dependencies
!pip install ninja packaging

# 1c. Install causal-conv1d from source
!pip install causal-conv1d --no-build-isolation

# 1d. Install mamba-ssm from source
!pip install mamba-ssm --no-build-isolation

# 1e. Install other required packages
!pip install wandb einops scipy tqdm tensorboard -q

In [ ]:
# 1f. Sanity check GPU and installations
import torch
print("PyTorch Version :", torch.__version__)
print("CUDA Available  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name :", torch.cuda.get_device_name(0))

try:
    from mamba_ssm import Mamba
    print("mamba_ssm       : OK (Import successful)")
except ImportError as e:
    print("mamba_ssm       : FAILED to import. Please check compilation logs above.")

## Step 2: Configure Dataset and Workspace
Set your training target (dataset and split) and specify the paths where your Kaggle inputs are located.

In [ ]:
# ==========================================
# USER CONFIGURATION
# ==========================================
# Target dataset: Choose from 'gtea', '50salads', 'breakfast'
DATASET = 'gtea'

# Split index:
# - GTEA: 1..4
# - 50Salads: 1..5
# - Breakfast: 1..4
SPLIT = 1

# Path where the ActSSM-DualToken repository was uploaded/attached as input
PROJECT_SRC = '/kaggle/input/datasets/muhammmadkamilkhan/actssm-dualtoken/ActSSM-DualToken'

# Path where the training dataset is attached (e.g., GTEA, 50Salads, Breakfast)
# Check the right sidebar in Kaggle -> Input files to verify this path!
DATASET_SRC = '/kaggle/input/datasets/asad1212/gtea-i3d/gtea'

# Destination in Kaggle working directory
PROJECT_DEST = '/kaggle/working/ActSSM'
# ==========================================

In [ ]:
import os
import shutil

# 2a. Copy repository code to working directory
if os.path.exists(PROJECT_DEST):
    shutil.rmtree(PROJECT_DEST)
shutil.copytree(PROJECT_SRC, PROJECT_DEST)
print(f"Project code copied to: {PROJECT_DEST}")

# 2b. Symlink or copy dataset to datasets/ folder
os.makedirs(f"{PROJECT_DEST}/datasets", exist_ok=True)
dst_folder = os.path.join(PROJECT_DEST, "datasets", DATASET.lower())

if os.path.exists(dst_folder):
    if os.path.islink(dst_folder):
        os.unlink(dst_folder)
    else:
        shutil.rmtree(dst_folder)

# Try symlinking (instant, saves disk space)
try:
    os.symlink(DATASET_SRC, dst_folder)
    print(f"Created symlink: {DATASET_SRC} -> {dst_folder}")
except Exception as e:
    print(f"Symlink failed ({e}). Falling back to copying...")
    shutil.copytree(DATASET_SRC, dst_folder)
    print(f"Copied dataset to: {dst_folder}")

# 2c. Verify dataset structure
print("\nVerifying dataset files:")
if os.path.exists(dst_folder):
    contents = os.listdir(dst_folder)
    print(f"Dataset root contains: {contents}")
    for item in ['features', 'groundTruth', 'splits', 'mapping.txt']:
        p = os.path.join(dst_folder, item)
        print(f" - {item:12s}: {'OK' if os.path.exists(p) else 'MISSING'}")
else:
    print("Error: Dataset folder does not exist!")

## Step 3: Weights & Biases (W&B) Setup
Choose between logging training progress online (requires your API key) or offline.

In [ ]:
import os
import wandb

# Option A: Online logging (uncomment and replace with your key)
# wandb.login(key='YOUR_WANDB_API_KEY')

# Option B: Offline logging (no internet/key needed, saved locally)
os.environ['WANDB_MODE'] = 'offline'
print("W&B Mode set to:", os.environ.get('WANDB_MODE', 'online'))

## Step 4: Load and Prepare Config File
We load the JSON configuration file from the repository, allow easy parameter adjustments (like changing the number of epochs or log frequency), and write the finalized configuration.

In [ ]:
import json

# Map dataset name to the appropriate config filename
config_name_map = {
    'gtea': 'GTEA.json',
    '50salads': '50salads.json',
    'breakfast': 'Breakfast.json'
}

config_file = config_name_map.get(DATASET.lower())
if not config_file:
    raise ValueError(f"Unknown dataset '{DATASET}'. Choose from 'gtea', '50salads', 'breakfast'.")

config_path = os.path.join(PROJECT_DEST, 'configs', config_file)

with open(config_path, 'r') as f:
    config = json.load(f)

# Ensure the root data directory points to the working datasets folder
config['root_data_dir'] = './datasets'

# Feel free to override any training parameters here
# Example:
# config['num_epochs'] = 10001
# config['batch_size'] = 4

print(f"Loaded config: {config_file}")
print(f"Dataset name : {config.get('dataset_name')}")
print(f"Epochs       : {config.get('num_epochs')}")
print(f"Batch Size   : {config.get('batch_size')}")
print(f"Learning Rate: {config.get('learning_rate')}")
print(f"Weight Decay : {config.get('weight_decay')}")
print(f"Postprocess  : {config.get('postprocess')}")

# Save the updated config back to the project folder
run_config_path = os.path.join(PROJECT_DEST, 'configs', 'run_config.json')
with open(run_config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f"\nFinal configuration written to: {run_config_path}")

## Step 5: Start Training
Run the training process using `main.py` with the prepared configuration.

In [ ]:
import sys
import subprocess

RESULT_TAG = f"actssm_{DATASET}_split{SPLIT}"

# Training CLI command
cmd_train = [
    sys.executable, 'main.py',
    '--config',     'configs/run_config.json',
    '--split',      str(SPLIT),
    '--result_dir', RESULT_TAG
]

print("Running command:", " ".join(cmd_train))
print(f"Working directory: {PROJECT_DEST}\n")

process = subprocess.Popen(
    cmd_train,
    cwd=PROJECT_DEST,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Print output in real-time
for line in process.stdout:
    print(line, end='', flush=True)

retcode = process.wait()
print(f"\nProcess exited with code {retcode}")

## Step 6: Test / Evaluation Only
Evaluate the best saved checkpoint model on the test split.

In [ ]:
# Run evaluation on the test split
cmd_test = [
    sys.executable, 'main.py',
    '--config',     'configs/run_config.json',
    '--split',      str(SPLIT),
    '--result_dir', RESULT_TAG,
    '--test'
]

print("Running test command:", " ".join(cmd_test))

process = subprocess.Popen(
    cmd_test,
    cwd=PROJECT_DEST,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()

## Step 7: Sequential Multi-Split Training
If you want to train all splits sequentially (e.g. split 1 to 4 for GTEA/Breakfast, or split 1 to 5 for 50Salads), uncomment and execute this cell.

In [ ]:
# # Uncomment below to run multi-split training:
#
# num_splits = 4 if DATASET.lower() in ['gtea', 'breakfast'] else 5
# for split_idx in range(1, num_splits + 1):
#     print("\n" + "="*65)
#     print(f" TRAINING SPLIT {split_idx} / {num_splits} FOR {DATASET.upper()}")
#     print("="*65)
#     
#     split_tag = f"actssm_{DATASET}_split{split_idx}"
#     cmd = [
#         sys.executable, 'main.py',
#         '--config',     'configs/run_config.json',
#         '--split',      str(split_idx),
#         '--result_dir', split_tag
#     ]
#     
#     process = subprocess.Popen(
#         cmd, cwd=PROJECT_DEST,
#         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
#         text=True, bufsize=1
#     )
#     for line in process.stdout:
#         print(line, end='', flush=True)
#     rc = process.wait()
#     print(f"Split {split_idx} completed with exit code {rc}")

## Step 8: Save and Summarize Results
Collect training outputs, logs, and checkpoints into a dedicated output folder for easy download from Kaggle.

In [ ]:
import os
import shutil
import glob

OUTPUT_DIR = '/kaggle/working/actssm_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Copy results from project result/ directory
result_base = os.path.join(PROJECT_DEST, 'result')
if os.path.exists(result_base):
    for item in os.listdir(result_base):
        src = os.path.join(result_base, item)
        dst = os.path.join(OUTPUT_DIR, item)
        if os.path.exists(dst):
            if os.path.islink(dst) or os.path.isfile(dst):
                os.unlink(dst)
            else:
                shutil.rmtree(dst)
        if os.path.isdir(src):
            shutil.copytree(src, dst)
            print(f"Copied directory: {item}")
        else:
            shutil.copy2(src, dst)
            print(f"Copied file: {item}")

# Summarize metrics from log files
print("\n" + "="*65)
print("METRIC LOGS SUMMARY")
print("="*65)
log_files = sorted(glob.glob(f"{OUTPUT_DIR}/**/log.txt", recursive=True))
for log_file in log_files:
    print(f"\n--- Log File: {os.path.relpath(log_file, OUTPUT_DIR)} ---")
    with open(log_file, 'r') as f:
        lines = f.readlines()
        # Print the last 25 lines of each log file (contains the final metrics tables)
        for line in lines[-25:]:
            print(line, end='')